# Text analysis examples
In this practicum, we learn to manipulate textual data and represent it numerically, moving from classic NLP to modern language models:

1. **Classic NLP** – tokenization, lemmatization, stop words and named entities (NLTK, spaCy)
2. **TF-IDF** – sparse bag-of-words representations
3. **Static word embeddings** – GloVe
4. **Contextual embeddings** – BERT and Sentence-BERT
5. **Classification** – comparing the representations above on the same task
6. **Large Language Models (LLMs)** – calling Gemini through the free Google AI Studio API for generation and zero/few-shot classification

**Colab tip:** before you start, select `Runtime → Change runtime type → T4 GPU`. Everything runs on CPU too, but the BERT section is much faster on a GPU. For section 6 you need a free Gemini API key: follow the instructions at the start of section 6 **before class**.

Throughout the notebook, **✏️ Your turn** cells ask you to modify and run code, and **🤔 Think** cells ask you to interpret the results. Write short answers in the notebook.

## Setup

Install the libraries Colab doesn't have (`-q` = quiet).

In [2]:
# Most packages are preinstalled on Colab; gensim, sentence-transformers and google-genai are not.
!pip install -q gensim sentence-transformers google-genai

Import everything up front. `nltk.download` fetches WordNet (used by the lemmatizer); `device` selects the GPU if one is available.

In [3]:
import re
import json
import numpy as np
import pandas as pds

import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer
import spacy

import gensim.downloader as api

import sklearn.model_selection as ms
import sklearn.feature_extraction.text as text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

import torch
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

nltk.download('wordnet', quiet=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cuda


Download the dataset (`-nc` skips the download if the file already exists).

In [4]:
!wget -nc -q https://raw.githubusercontent.com/msoley/DSCI549/master/In-class%20exercises/Practicum4/tweet_global_warming.csv
dataset = './tweet_global_warming.csv'
#source https://www.figure-eight.com/data-for-everyone/

## 1. Classic NLP
### NLTK
In this section we tokenize a sentence and lemmatize a word.

**Tokenization** splits text into units (tokens). The regex `\w+` keeps runs of letters/digits, so punctuation is dropped.

In [5]:
tokenizer = RegexpTokenizer(r'\w+')
example_text = "Penguins are flightless birds."
tokenizer.tokenize(example_text)

['Penguins', 'are', 'flightless', 'birds']

**Lemmatization** maps a word to its dictionary form (*brought → bring*). The part of speech matters: `pos='v'` means "treat it as a verb".

In [6]:
lemmatizer = WordNetLemmatizer()
lemmatizer.lemmatize('brought', pos='v')

'bring'

### spaCy
spaCy is a popular NLP toolbox that runs a whole pipeline (tokenizer, lemmatizer, tagger, named entity recognizer, ...) in one call.

`nlp(text)` runs the whole pipeline; each token then carries its lemma and a stop-word flag.

In [7]:
nlp = spacy.load("en_core_web_sm")
doc = nlp(example_text)
print(example_text)
print()
for token in doc:
    print(token.text, token.lemma_, token.is_stop)

Penguins are flightless birds.

Penguins penguin False
are be True
flightless flightless False
birds bird False
. . False


**Stop words** are very frequent function words (*the, is, of*) that carry little topical meaning; bag-of-words methods often drop them.

In [8]:
#  list of default stop words in spaCy english model
stopwords = nlp.Defaults.stop_words
print(len(stopwords))
sorted(stopwords)[:30]

326


["'d",
 "'ll",
 "'m",
 "'re",
 "'s",
 "'ve",
 'a',
 'about',
 'above',
 'across',
 'after',
 'afterwards',
 'again',
 'against',
 'all',
 'almost',
 'alone',
 'along',
 'already',
 'also',
 'although',
 'always',
 'am',
 'among',
 'amongst',
 'amount',
 'an',
 'and',
 'another',
 'any']

> **🤔 Think:** Are *not*, *no* and *never* stop words? Why would removing them be a problem for our task (detecting tweets that deny global warming)?

In [9]:
'not' in stopwords, 'no' in stopwords, 'never' in stopwords

(True, True, True)

### Data loading and preprocessing
Our dataset contains tweets about global warming, labeled by whether the tweet suggests global warming is happening (`existence`: Yes/No).

The file is not UTF-8, so we specify the encoding; `on_bad_lines='warn'` skips malformed rows.

In [10]:
data = pds.read_csv(filepath_or_buffer=dataset, encoding="ISO-8859-1", on_bad_lines='warn')
print(data.shape)
data.head()

(6090, 3)


,tweet,existence,existence.confidence
0,Global warming report urges governments to act...,Yes,1.0000
1,Fighting poverty and global warming in Africa ...,Yes,1.0000
2,Carbon offsets: How a Vatican forest failed to...,Yes,0.8786
3,Carbon offsets: How a Vatican forest failed to...,Yes,1.0000
4,URUGUAY: Tools Needed for Those Most Vulnerabl...,Yes,0.8087


Check the label distribution. Note the inconsistent labels (Yes/Y, No/N) and missing values (NaN): real data is messy.

In [11]:
data['existence'].value_counts(dropna=False)

,count
existence,
Y,2554
NaN,1865
N,1053
Yes,557
No,61


> **🤔 Think:** Is the dataset balanced? What accuracy would a classifier that always predicts "Yes" get?

**Normalization** with regular expressions: replace URLs and @handles with placeholders (each one is rare and uninformative on its own), then collapse whitespace.

In [12]:
tweets = data['tweet'].tolist()
# replace links
tweet_normed = [re.sub(r"http:\/\/.*", "[link]", x) for x in tweets]
print(tweets[1091])
print(tweet_normed[1091])
# replace twitter handles
tweet_normed = [re.sub(r'@[A-Za-z0-9_-]* ', '[twitter_handle] ', x) for x in tweet_normed]
print(tweets[31])
print(tweet_normed[31])
# remove extra spaces
tweet_normed = [re.sub(r"\s+", ' ', x) for x in tweet_normed]
print(tweet_normed[31])
data['tweet_normed'] = tweet_normed

LIVE TONIGHT: Fellow Heather Rogers joins Climate Change Expert James Hansen to discuss REAL solutions to Climate... http://bit.ly/aT8CgK
LIVE TONIGHT: Fellow Heather Rogers joins Climate Change Expert James Hansen to discuss REAL solutions to Climate... [link]
@New_federalists  i have it on good auth tht global warming also causes toe fungus.  We R all fortunate tht thr IS no global warming! #tcot
[twitter_handle]  i have it on good auth tht global warming also causes toe fungus.  We R all fortunate tht thr IS no global warming! #tcot
[twitter_handle] i have it on good auth tht global warming also causes toe fungus. We R all fortunate tht thr IS no global warming! #tcot


> **✏️ Your turn:** The link regex `http:\/\/.*` is greedy (it deletes everything after the link) and misses `https` links. Look at the examples below and fix it (hint: `\S+`).

In [13]:
# examples with text after the link
[t for t in tweets if re.search(r'http\S+ \w', t)][:3]

['Even the generals know climate change is going to screw us. http://tinyurl.com/2urnnvo Clean energy bill, anyone? #climate',
 'CLIMATE CHANGE: Forests Not for Absorbing Carbon, Say Activists http://bit.ly/bRHMYi IPSAmazon',
 'Kuna Indians Prepare For Relocation As Traditional Homes Sink Due To Climate Change http://bit.ly/cEs5qN by @indigenousissue']

**Named entity recognition (NER)** finds spans such as people, organizations and places. `spacy.explain('GPE')` describes a label.

In [14]:
# named entities
doc = nlp(tweet_normed[1091])
print(tweet_normed[1091])
print()
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

LIVE TONIGHT: Fellow Heather Rogers joins Climate Change Expert James Hansen to discuss REAL solutions to Climate... [link]

Heather Rogers 21 35 PERSON
James Hansen 64 76 PERSON


The same tweet, lowercased:

In [15]:
doc = nlp(tweet_normed[1091].lower())
print(tweet_normed[1091].lower())
print()
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

live tonight: fellow heather rogers joins climate change expert james hansen to discuss real solutions to climate... [link]

tonight 5 12 TIME
james hansen 64 76 PERSON


*Lowercasing hurts NER: capitalization is an important cue. Off-the-shelf tools are trained on news text and are not perfect on tweets; consider using TweetNLP!*
https://tweetnlp.org/

## 2. TF-IDF
TF-IDF represents a document as a (sparse) vector with one dimension per vocabulary word:

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \log\frac{1 + N}{1 + \text{df}(t)} + 1$$

* **tf(t, d)**: how often term *t* occurs in document *d*
* **df(t)**: how many of the *N* documents contain *t*

Words that occur in every document (e.g., "the") get a low weight; words that are frequent in one document but rare overall get a high weight. (The formula above is scikit-learn's smoothed default; each row is then L2-normalized.)

Let's start with a toy corpus.

`fit_transform` learns the vocabulary and idf values and returns the tf-idf matrix (rows = documents, columns = words). Note that "a" is missing: the default tokenizer ignores one-character tokens.

In [16]:
toy_corpus = [
    "the penguin is a flightless bird",
    "the eagle is a bird that can fly",
    "the stock market fell today",
    "the market for penguin plush toys is booming",
]
toy_tf = text.TfidfVectorizer()
X_toy = toy_tf.fit_transform(toy_corpus)
pds.DataFrame(X_toy.toarray(), columns=toy_tf.get_feature_names_out(), index=[f'doc{i}' for i in range(len(toy_corpus))]).round(2)

,bird,booming,can,eagle,fell,flightless,fly,for,is,market,penguin,plush,stock,that,the,today,toys
doc0,0.46,0.00,0.00,0.00,0.00,0.58,0.00,0.00,0.37,0.00,0.46,0.00,0.00,0.00,0.31,0.00,0.00
doc1,0.34,0.00,0.43,0.43,0.00,0.00,0.43,0.00,0.28,0.00,0.00,0.00,0.00,0.43,0.23,0.00,0.00
doc2,0.00,0.00,0.00,0.00,0.51,0.00,0.00,0.00,0.00,0.40,0.00,0.00,0.51,0.00,0.26,0.51,0.00
doc3,0.00,0.41,0.00,0.00,0.00,0.00,0.00,0.41,0.26,0.32,0.32,0.41,0.00,0.00,0.21,0.00,0.41


In [17]:
# idf for each word: "the" and "is" appear everywhere -> lowest idf
pds.Series(toy_tf.idf_, index=toy_tf.get_feature_names_out()).sort_values().round(2)

,0
the,1.00
is,1.22
bird,1.51
penguin,1.51
market,1.51
booming,1.92
can,1.92
eagle,1.92
for,1.92
fly,1.92


**Cosine similarity** measures the angle between two vectors: 1 = same direction, 0 = no shared words.

In [18]:
# documents that share informative words are more similar
pds.DataFrame(cosine_similarity(X_toy), index=[f'doc{i}' for i in range(4)], columns=[f'doc{i}' for i in range(4)]).round(2)

,doc0,doc1,doc2,doc3
doc0,1.00,0.33,0.08,0.31
doc1,0.33,1.00,0.06,0.12
doc2,0.08,0.06,1.00,0.19
doc3,0.31,0.12,0.19,1.00


> **🤔 Think:** Why is doc0 more similar to doc3 than to doc2, even though doc0 shares "the" and "is" with both?

**Limitation:** TF-IDF only matches exact words. "An eagle can fly" and "Birds soar in the sky" have zero similarity even though they are about the same thing. Embeddings fix this.

In [19]:
cosine_similarity(toy_tf.transform(["an eagle can fly"]), toy_tf.transform(["birds soar in the sky"]))

array([[0.]])

### TF-IDF on tweets
Now on the real data. We keep only tweets with a Yes/No label and drop duplicates (retweets), so that the same tweet does not end up in both training and test sets.

In [20]:
labeled = data[data['existence'].isin(["Yes", "Y", "No", "N"])].copy()
labeled = labeled.drop_duplicates(subset='tweet_normed')
# we encode climate deniers as 1
labeled['label'] = labeled['existence'].isin(["No", "N"]).astype(int)
print(labeled['label'].value_counts())

train_df, test_df = ms.train_test_split(labeled, test_size=0.2, stratify=labeled['label'], random_state=42)
train_set, y_train = train_df['tweet_normed'].tolist(), train_df['label'].values
test_set, y_test = test_df['tweet_normed'].tolist(), test_df['label'].values
len(train_set), len(test_set)

label
0    2280
1     969
Name: count, dtype: int64


(2599, 650)

We **fit** the vectorizer on the training set only and **transform** the test set with the same vocabulary; fitting on the test set would leak information. `ngram_range=(1, 2)` adds word pairs (bigrams); `min_df=2` drops terms that appear in fewer than 2 tweets.

In [21]:
tf = text.TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
x_train_tfidf = tf.fit_transform(train_set)
x_test_tfidf = tf.transform(test_set)
print(x_train_tfidf.shape)
print(x_test_tfidf.shape)

(2599, 5450)
(650, 5450)


The matrix is stored **sparse**: only non-zero entries are kept in memory.

In [22]:
#estimate sparsity
p = 100 * x_train_tfidf.nnz / float(x_train_tfidf.shape[0] * x_train_tfidf.shape[1])
print(f"Each sample has ~{p:.2f}% non-zero features.")

Each sample has ~0.25% non-zero features.


## 3. Static word embeddings (GloVe)
Word embeddings map each word to a dense vector (here 100 dimensions) learned from co-occurrence statistics in a large corpus, so that words appearing in similar contexts get similar vectors.

`gensim.downloader` provides pretrained vectors. GloVe was trained on Wikipedia and newswire text.

In [23]:
# download the model (~130MB). glove-twitter-25/50/100/200 are trained on tweets instead.
glove = api.load("glove-wiki-gigaword-100")
glove['cat'].shape, glove['cat'][:10]

[==================================================] 100.0% 128.1/128.1MB downloaded


((100,),
 array([ 0.23088 ,  0.28283 ,  0.6318  , -0.59411 , -0.58599 ,  0.63255 ,
         0.24402 , -0.14108 ,  0.060815, -0.7898  ], dtype=float32))

Nearest neighbors by cosine similarity:

In [24]:
glove.most_similar("cat", topn=10)

[('dog', 0.8798074722290039),
 ('rabbit', 0.7424427270889282),
 ('cats', 0.732300341129303),
 ('monkey', 0.7288709878921509),
 ('pet', 0.719014048576355),
 ('dogs', 0.7163872718811035),
 ('mouse', 0.6915250420570374),
 ('puppy', 0.6800068020820618),
 ('rat', 0.6641027331352234),
 ('spider', 0.6501135230064392)]

In [25]:
# similarity is the cosine between vectors
print(glove.similarity('warming', 'climate'), glove.similarity('warming', 'banana'))
# remember TF-IDF said eagle/fly and birds/soar were unrelated:
print(glove.similarity('eagle', 'birds'), glove.similarity('fly', 'soar'))

0.79910123 0.17134835
0.28470036 0.30280575


### Analogies: vector arithmetic
`king - man + woman ≈ ?`

In [26]:
glove.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)

[('queen', 0.7698540687561035),
 ('monarch', 0.6843381524085999),
 ('throne', 0.6755736470222473)]

In [27]:
print(glove.most_similar(positive=['paris', 'italy'], negative=['france'], topn=3))
print(glove.most_similar(positive=['walking', 'swam'], negative=['walked'], topn=3))

[('rome', 0.8189547061920166), ('milan', 0.7376196980476379), ('naples', 0.7117615342140198)]
[('swimming', 0.7067671418190002), ('swim', 0.7048406600952148), ('biking', 0.6536761522293091)]


> **✏️ Your turn:** Find one analogy that works and one that fails. Why do you think the second one fails?

In [41]:
print(glove.most_similar(positive=['monday', 'friday'], negative=['weekend'], topn=3))
print(glove.most_similar(positive=['hockey', 'boston'], negative=['game'], topn=3))

[('thursday', 0.8905558586120605), ('wednesday', 0.8877512216567993), ('tuesday', 0.8874239325523376)]
[('montreal', 0.6494169235229492), ('toronto', 0.6457923650741577), ('winnipeg', 0.6341705322265625)]


Embeddings also absorb **biases** present in the training text. Try some analogies yourself:

In [29]:
print(glove.most_similar(positive=['doctor', 'she'], negative=['he'], topn=5))
print(glove.most_similar(positive=['programmer', 'she'], negative=['he'], topn=5))

[('nurse', 0.7816784977912903), ('mother', 0.7166498303413391), ('woman', 0.6973024010658264), ('pregnant', 0.6716630458831787), ('patient', 0.6694736480712891)]
[('programmers', 0.5739936828613281), ('animator', 0.5649070143699646), ('stylist', 0.5622987747192383), ('technician', 0.5470567345619202), ('software', 0.5230430960655212)]


> **🤔 Think:** Where do these associations come from? Why would they be a problem if such embeddings were used, for example, to rank job applicants?

**Limitation:** a static embedding has *one* vector per word regardless of context. "bank" gets the same vector in "river bank" and "bank account" – its neighbors are a mix of both senses:

In [49]:
glove.most_similar('game', topn=10)

[('games', 0.8644920587539673),
 ('play', 0.8316268920898438),
 ('season', 0.7733193635940552),
 ('player', 0.7579817175865173),
 ('players', 0.7287797331809998),
 ('match', 0.7282912135124207),
 ('scoring', 0.720136284828186),
 ('playing', 0.7145521640777588),
 ('playoffs', 0.7068414092063904),
 ('team', 0.7012254595756531)]

### From word vectors to tweet vectors
A simple way to represent a whole tweet is to average the vectors of its words.

In [31]:
def glove_embed(texts):
    out = np.zeros((len(texts), glove.vector_size), dtype=np.float32)
    for i, doc in enumerate(nlp.pipe(texts, disable=['parser', 'ner'])):
        vs = [glove[t.lower_] for t in doc if t.lower_ in glove and not t.is_stop and not t.is_punct]
        if vs:
            out[i] = np.mean(vs, axis=0)
    return out

x_train_glove = glove_embed(train_set)
x_test_glove = glove_embed(test_set)
x_train_glove.shape

(2599, 100)

> **🤔 Think:** Averaging ignores word order and negation: write two tweets with almost the same words but opposite meanings.

## 4. Contextual embeddings (BERT)
BERT is a Transformer encoder. Unlike GloVe, it produces a **different vector for each token depending on the whole sentence**.

### Subword tokenization
BERT uses WordPiece tokenization: rare words are split into pieces (`##` marks a continuation), so there is no out-of-vocabulary problem.

In [32]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
for s in ["penguins are flightless birds.", "#ClimateChange denialism is unbelievable"]:
    print(s, '->', bert_tokenizer.tokenize(s))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

penguins are flightless birds. -> ['penguins', 'are', 'flight', '##less', 'birds', '.']
#ClimateChange denialism is unbelievable -> ['#', 'climate', '##chang', '##e', 'denial', '##ism', 'is', 'unbelievable']


Load the pretrained model. `.eval()` turns off dropout since we are not training.

In [33]:
bert = BertModel.from_pretrained('bert-base-uncased').to(device).eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### The same word, different vectors
Let's get BERT's vector for "bank" in different sentences and compare them.

`word_vector` runs BERT on a sentence (`torch.no_grad()` skips gradient bookkeeping) and returns the hidden state at the position of `word`.

In [34]:
def word_vector(sentence, word):
    enc = bert_tokenizer(sentence, return_tensors='pt').to(device)
    with torch.no_grad():
        hidden = bert(**enc).last_hidden_state[0]
    toks = bert_tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    return hidden[toks.index(word)].cpu().numpy()

bank_sentences = [
    "I deposited the check at the bank.",
    "The bank approved my loan application.",
    "We had a picnic on the bank of the river.",
    "The fisherman sat on the muddy bank.",
]
bank_vecs = np.array([word_vector(s, 'bank') for s in bank_sentences])
pds.DataFrame(cosine_similarity(bank_vecs), index=bank_sentences, columns=range(4)).round(2)

,0,1,2,3
I deposited the check at the bank.,1.00,0.81,0.53,0.52
The bank approved my loan application.,0.81,1.00,0.44,0.43
We had a picnic on the bank of the river.,0.53,0.44,1.00,0.80
The fisherman sat on the muddy bank.,0.52,0.43,0.80,1.00


The two "money" banks are much more similar to each other than to the two "river" banks: BERT disambiguates word senses from context.

> **✏️ Your turn:** Try another ambiguous word (e.g., "apple", "bat", "spring") in a few sentences. Does BERT separate the senses? (The word must be a single WordPiece token.)

In [54]:
sents = [
    "I bought a apple",
    "Its apple season",
]
vecs = np.array([word_vector(s, 'apple') for s in sents])
pds.DataFrame(cosine_similarity(vecs), index=sents).round(2)

,0,1
I bought a apple,1.00,0.46
Its apple season,0.46,1.00


### Sentence embeddings
To get one vector per sentence we can average (mean-pool) the token vectors. Models such as **Sentence-BERT** are fine-tuned specifically so that these sentence vectors capture meaning; `all-MiniLM-L6-v2` is a small, fast one.

`encode` returns one 384-d vector per sentence.

In [55]:
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)
sentences = ["An eagle can fly", "Birds soar in the sky", "The stock market fell today"]
emb = sbert.encode(sentences)
print(emb.shape)
pds.DataFrame(cosine_similarity(emb), index=sentences, columns=sentences).round(2)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)


,An eagle can fly,Birds soar in the sky,The stock market fell today
An eagle can fly,1.00,0.5,0.06
Birds soar in the sky,0.50,1.0,0.10
The stock market fell today,0.06,0.1,1.00


Compare with TF-IDF above: "An eagle can fly" and "Birds soar in the sky" share no words, yet their sentence embeddings are similar.

### Semantic search over tweets
Embed all tweets once, then find the tweets closest in meaning to a query.

With `normalize_embeddings=True` every vector has length 1, so a dot product equals the cosine similarity.

In [56]:
x_train_sbert = sbert.encode(train_set, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
x_test_sbert = sbert.encode(test_set, batch_size=64, normalize_embeddings=True)
x_train_sbert.shape

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

(2599, 384)

In [57]:
def search(query, k=5):
    q = sbert.encode([query], normalize_embeddings=True)
    scores = (x_train_sbert @ q.T).ravel()
    for i in scores.argsort()[::-1][:k]:
        print(f"{scores[i]:.2f}  {train_set[i]}")

search("scientists faked the temperature data")

0.61  Climate change - data manipulated for 20 years! [link]
0.59  computer forensics experts: are there still any idiots out there that belive global warming is true? [link]
0.58  That's why he removed the data that disagreed with his global warming theory. [link]
0.58  Global Warming Monitoring Needs to Find 'Missing Heat', say Scientists|by David Adam Experts need to beef up ways .. [link]
0.58  Global Warming: Climate Scientists Investigate Lost Heat|A new article in 'Science' explores the mystery of the .. [link]


In [58]:
search("polar bears are losing their habitat")

0.51  PHOTO OF THE DAY: A polar bear in Nunavut. It is listed as threatened & faces critical danger due to global warming. [link]
0.50  Wolverine No Match for Climate Change: Forget polar bears or penguins. The charismatic megafauna suffering most ... [link]
0.47  Protect Wildlife Habitat from Climate Change: [link]
0.47  [twitter_handle] -I'd spend money on moving the over heated polar bears to east coast. Would be cheaper that FUNDING GLOBAL WARMING
0.44  6 of 25 known Great Basin pika populations were lost by 2003 primarily due to climate change. More have likely disappeared since.


> **✏️ Your turn:** Write two queries of your own and inspect the results. Compare with the keyword search below. When does semantic search do better?

In [59]:
search("your query here")

# keyword search for comparison
[t for t in train_set if 'hoax' in t.lower()][:5]

0.28  [link]
0.26  . [link]
0.22  :) [link]
0.22  [twitter_handle] re: where is the global warming? If anyone would have found it, it would be you. where havent you looked yet?
0.22  [twitter_handle] [link]


['It Seems The Global Warming Hoax Is Starting To Sink In. What Will Th',
 "[twitter_handle] I'm prepared to get on a plane & punch my first FB friend who points to this storm as evidence of global warming being a hoax.",
 'India is first country to probe climate change hoax. Wake up America. Al Gore and his greenies are a fraud.',
 'The global warming hoax? Houston expert talks climate alarmist pollution - CultureMap - [link]',
 '[twitter_handle] you are so on target, Bullshit is even better than the climate change hoax & Gore /Climate change is bullshit #leadersdebate #fact']

## 5. Classification: comparing representations
We train the same classifier (logistic regression) on each representation and compare the F1 score for the minority class (deniers).

`GridSearchCV` tries each value of `C` (inverse regularization strength) with 5-fold cross-validation **on the training set** and keeps the best; the test set is used only once at the end. We report F1 for the denier class because the classes are imbalanced (accuracy rewards always predicting "Yes"). `class_weight='balanced'` gives more weight to errors on the minority class.

In [60]:
results = {}

def evaluate(name, clf, x_tr, x_te):
    clf.fit(x_tr, y_train)
    pred = clf.predict(x_te)
    results[name] = {'accuracy': accuracy_score(y_test, pred), 'F1 (denier)': f1_score(y_test, pred)}
    print(name, results[name])
    return clf

grid = {"C": np.logspace(-2, 3, 6)}
def logreg():
    return ms.GridSearchCV(LogisticRegression(max_iter=2000, class_weight='balanced'), param_grid=grid, scoring='f1')

lr_tfidf = evaluate('TF-IDF + LogReg', logreg(), x_train_tfidf, x_test_tfidf)
lr_glove = evaluate('GloVe avg + LogReg', logreg(), x_train_glove, x_test_glove)
lr_sbert = evaluate('Sentence-BERT + LogReg', logreg(), x_train_sbert, x_test_sbert)

TF-IDF + LogReg {'accuracy': 0.74, 'F1 (denier)': 0.6114942528735632}
GloVe avg + LogReg {'accuracy': 0.7476923076923077, 'F1 (denier)': 0.6238532110091743}
Sentence-BERT + LogReg {'accuracy': 0.7584615384615384, 'F1 (denier)': 0.6357308584686775}


In [61]:
pds.DataFrame(results).T.round(3)

,accuracy,F1 (denier)
TF-IDF + LogReg,0.740,0.611
GloVe avg + LogReg,0.748,0.624
Sentence-BERT + LogReg,0.758,0.636


> **🤔 Think:** Which representation works best? Is it surprising that a 384-d sentence embedding can compete with thousands of TF-IDF features?

Logistic regression is interpretable: each feature has one weight; positive weights push towards "denier".

In [62]:
# which words push the TF-IDF logistic regression towards "denier"?
vocab = tf.get_feature_names_out()
coefs = lr_tfidf.best_estimator_.coef_.ravel()
print("denier:  ", list(vocab[coefs.argsort()[::-1][:15]]))
print("believer:", list(vocab[coefs.argsort()[:15]]))

denier:   ['gore', 'global warming', 'hoax', 'al', 'scam', 'global', 'snow', 'tcot', 'warming', 'al gore', 'twitter_handle', 'scandal', 'utah', 'conspiracy', 'scientist']
believer: ['link', 'warming link', 'news', 'change', 'effects', 'world', 'climate change', 'green', 'climate', 'national', 'post', 'reduce', 'impacts', 'stop global', 'allergies']


In [63]:
#let's test on made up samples
test_texts = [
    "I don't believe in climate change",
    "Global warming is endangering our livelihoods.",
    "The whole country is freezing; this must be due to global warming.",
    "The whole climate crisis is not only Fake News, it's Fake Science.",
]
pds.DataFrame({
    'TF-IDF LR': lr_tfidf.predict(tf.transform(test_texts)),
    'GloVe LR': lr_glove.predict(glove_embed(test_texts)),
    'SBERT LR': lr_sbert.predict(sbert.encode(test_texts, normalize_embeddings=True)),
}, index=test_texts)

,TF-IDF LR,GloVe LR,SBERT LR
I don't believe in climate change,0,0,0
Global warming is endangering our livelihoods.,1,0,0
The whole country is freezing; this must be due to global warming.,1,0,1
"The whole climate crisis is not only Fake News, it's Fake Science.",0,1,1


0: believer 1: denier

Notice the mistakes: "I don't believe in climate change" is classified as a *believer* by every model, because bag-of-words and averaged embeddings largely ignore negation, and the phrase "global warming" itself is more frequent in denier tweets in this dataset. Keep these examples in mind for the LLM section.

> **✏️ Your turn:** Add a few sentences of your own to `test_texts` (e.g., sarcasm, negation) and re-run the cell. Which model is most robust?

In [ ]:
# your code here

## 6. Large Language Models (Gemini API)
LLMs are Transformer **decoders** trained to predict the next token, then tuned to follow instructions. Instead of running one locally, we call Google's **Gemini Flash-Lite** model through the free tier of the Gemini API.

### Get a free Gemini API key (do this before class, ~5 minutes)
You need an API key so that Google knows who is sending requests. The free tier does not require a credit card.

**Step 1: Sign in to Google AI Studio**
1. Open [https://aistudio.google.com](https://aistudio.google.com).
2. Sign in with a **personal Google account (e.g., @gmail.com)**. ⚠️ **Your @usc.edu account will not work**: USC does not give access to AI Studio. If you do not have a personal Google account, create one for free at [accounts.google.com/signup](https://accounts.google.com/signup).

   *Tip:* AI Studio may automatically sign you in with your USC account if you are logged into it in the browser. Use the account switcher (top right) or an incognito/private window to make sure you are using your personal account.
3. The first time, accept the terms of service. (You must be 18 or older and in a [supported country](https://ai.google.dev/gemini-api/docs/available-regions).)

**Step 2: Create a key**
1. Click **Get API key** in the left menu (or go directly to [https://aistudio.google.com/apikey](https://aistudio.google.com/apikey)).
2. Click **Create API key**. If you are asked to choose a Google Cloud project, pick the default one or let AI Studio create a new project for you.
3. Copy the key (a long string starting with `AIza...`).

**Step 3: Store the key in Colab Secrets**
1. In this Colab notebook, click the **key icon 🔑 (Secrets)** in the left sidebar.
2. Click **Add new secret**. Set **Name** to `GEMINI_API_KEY` (exactly this) and paste your key into **Value**.
3. Turn on the **Notebook access** toggle for this secret.

It does not matter which Google account you use for Colab itself: a key created with your personal account works in Colab under your USC account too. Secrets are stored in your Colab account, not in the notebook, so you only do this once and they are not included when you share or submit the notebook.

**Rules for API keys**
* **Never paste your key directly into a code cell**, a GitHub repository, Slack or an assignment submission. Anyone with your key can use your quota.
* If a key leaks, delete it on the [API keys page](https://aistudio.google.com/apikey) and create a new one.
* Do not enable billing on the project: the free tier is enough for this practicum. It allows a limited number of requests per minute and per day (see [rate limits](https://ai.google.dev/gemini-api/docs/rate-limits)), so the code below sends many tweets in one request.

**Troubleshooting**
| Error | Fix |
|---|---|
| `SecretNotFoundError` | The secret name is not exactly `GEMINI_API_KEY`. |
| `NotebookAccessError` | Turn on *Notebook access* for the secret in the 🔑 panel. |
| `400 API key not valid` | Copy the key again; make sure there are no extra spaces. |
| "This service is not available for your account" in AI Studio | You are signed in with your USC account; switch to a personal Google account. |
| `403 PERMISSION_DENIED` | The key was created with an account or in a region that cannot use the API; create it with a personal Gmail account. |
| `429 RESOURCE_EXHAUSTED` | You hit the rate limit: wait a minute (or until tomorrow for the daily limit). |
| `404 model not found` | The model name changed; pick a current Flash-Lite model from the [models page](https://ai.google.dev/gemini-api/docs/models). |

Read the key from Colab Secrets, create a client and send a test request.

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel
import os, time

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except ImportError:                      # running outside Colab
    api_key = os.environ.get('GEMINI_API_KEY') or __import__('getpass').getpass('Gemini API key: ')

client = genai.Client(api_key=api_key)
llm_name = "gemini-3.1-flash-lite"       # see https://ai.google.dev/gemini-api/docs/models for current models

# check that the key works
print(client.models.generate_content(model=llm_name, contents="Say hello to the DSCI 549 class in five words.").text)

A small helper that we use for every call: `temperature` controls randomness, `system` sets a system instruction and `schema` asks for JSON that matches a Pydantic class (returned as Python objects). It waits and retries when we hit the rate limit.

In [ ]:
def generate(prompt, temperature=0.0, system=None, schema=None, retries=3):
    config = types.GenerateContentConfig(temperature=temperature, system_instruction=system)
    if schema is not None:               # ask for JSON that follows a schema
        config.response_mime_type = 'application/json'
        config.response_schema = schema
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=llm_name, contents=prompt, config=config)
            return response.parsed if schema is not None else response.text.strip()
        except genai.errors.APIError as e:
            if e.code == 429 and attempt < retries - 1:   # rate limit: wait and retry
                print('Rate limited, waiting 30s...')
                time.sleep(30)
            else:
                raise

print(generate("In one sentence, what is the difference between TF-IDF and word embeddings?"))

### Temperature
Generation picks one token at a time from a probability distribution over the vocabulary. **Temperature** controls how we pick: low temperature ≈ always take the most likely token (repeatable answers); high temperature = more random/creative.

In [ ]:
prompt = "Write a one-sentence tweet about global warming."
for t in [0.0, 1.0, 2.0]:
    print(f"--- temperature={t}")
    for _ in range(2):
        print(generate(prompt, temperature=t))

> **🤔 Think:** What happens at temperature 2.0? Which temperature would you use for classification, and why?

### Zero-shot classification
Instead of training a classifier, we can simply *ask* the LLM. No training data needed!

In [ ]:
zero_shot_prompt = '''Here is a tweet about global warming:
"{tweet}"

Does this tweet suggest that global warming is happening? Answer with only "Yes" or "No".'''

def llm_classify(tweet):
    answer = generate(zero_shot_prompt.format(tweet=tweet))
    return 1 if answer.lower().startswith('no') else 0   # 1 = denier

for t in test_texts:
    print(llm_classify(t), t)

> **🤔 Think:** Does the LLM handle "I don't believe in climate change" correctly, unlike the classifiers in section 5? Why?

### Evaluating on the test set
To stay within the free-tier limits, we send many tweets in **one request** and ask for a JSON answer that follows a schema (*structured output*), so the reply can be parsed reliably.

In [ ]:
class Prediction(BaseModel):
    id: int
    answer: str          # "Yes" or "No"

def llm_classify_batch(tweets, instructions, batch_size=50):
    preds = []
    for start in range(0, len(tweets), batch_size):
        batch = tweets[start:start + batch_size]
        listing = '\n'.join(f'{i}: {t}' for i, t in enumerate(batch))
        out = generate(f'{instructions}\n\nTweets:\n{listing}', schema=list[Prediction])
        answers = {p.id: p.answer for p in out}
        preds += [1 if answers.get(i, 'Yes').lower().startswith('no') else 0 for i in range(len(batch))]
    return preds

zero_shot_instructions = '''For each tweet below, answer "Yes" if the tweet suggests that global warming is happening,
and "No" otherwise. Return one answer per tweet id.'''

# a balanced sample of 100 test tweets
sample = pds.concat([test_df[test_df.label == 0].sample(50, random_state=0),
                     test_df[test_df.label == 1].sample(50, random_state=0)])
llm_pred = llm_classify_batch(sample['tweet_normed'].tolist(), zero_shot_instructions)

sample_idx = [test_set.index(t) for t in sample['tweet_normed']]
sbert_pred = lr_sbert.predict(x_test_sbert[sample_idx])
print('Zero-shot LLM          F1 (denier): %.3f' % f1_score(sample['label'], llm_pred))
print('Sentence-BERT + LogReg F1 (denier): %.3f' % f1_score(sample['label'], sbert_pred))

In [ ]:
# look at where the LLM and the labels disagree. Are the labels always right?
sample['llm'] = llm_pred
sample[sample['llm'] != sample['label']][['tweet_normed', 'existence', 'llm']].head(10)

> **🤔 Think:** How many disagreements are LLM mistakes, and how many look like labeling errors or ambiguous tweets?

### Few-shot prompting
We can show the model a few labeled examples in the prompt (*in-context learning*). Note that the model's weights do not change.

In [ ]:
few_shot_instructions = zero_shot_instructions + '''

Examples:
"Fighting poverty and global warming in Africa [link]" -> Yes
"Snowing in April. So much for global warming." -> No
"Scientists say Arctic ice is melting faster than predicted [link]" -> Yes
"Global warming is the biggest hoax ever, follow the money" -> No'''

llm_pred_fs = llm_classify_batch(sample['tweet_normed'].tolist(), few_shot_instructions)
print('Few-shot LLM F1 (denier): %.3f' % f1_score(sample['label'], llm_pred_fs))

> **✏️ Your turn:** Rewrite the instructions (e.g., define the labels more precisely) and measure the effect on F1. Each run uses 2 requests of your free quota.

In [ ]:
my_instructions = zero_shot_instructions  # edit me

# llm_pred_mine = llm_classify_batch(sample['tweet_normed'].tolist(), my_instructions)
# print('My prompt F1 (denier): %.3f' % f1_score(sample['label'], llm_pred_mine))

### Things to keep in mind
* Outputs depend on the exact wording of the prompt. Always evaluate on labeled data, as we did above, instead of trusting a few examples.
* Your data is sent to an external service: think about privacy before sending sensitive text to an API.
* LLM APIs cost money (or quota) per token and are much slower than a logistic regression on embeddings: a trained small model is often the better choice when you have labeled data.